# 02 — جودة البيانات وتنظيف النص

**الجزء الأول (الأقسام 1–8): جودة البيانات.** كل قسم يمشي بنفس الترتيب: المشكلة، الطريقة، الأرقام والأمثلة، السبب، القرار. القرارات النهائية مكتوبة في القاموس `DECISIONS` في `quality.py`، وهي نفسها التي تستخدمها `apply_quality_fixes`.

**الجزء الثاني (الأقسام 9–13): تنظيف النص (`preprocess.py`).** أثر كل خطوة تنظيف على حدة، ثم المقارنة بين `light` و`heavy`، ثم مراجعة الكلمات الشائعة.

In [ ]:
import re
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from common import Columns, DATA_DIR, RANDOM_STATE, load_data
from quality import DECISIONS, QualityFixer, arabic_ratio

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
print("DATA_DIR =", DATA_DIR)

In [ ]:
train, val, test, label_map = load_data()
splits = {"train": train, "val": val, "test": test}
fixer = QualityFixer()

for name, df in splits.items():
    print(f"{name:<6} rows={len(df):>6}")

## 1. القيم الفارغة والنصوص القصيرة

**المشكلة:** السؤال الفارغ أو شبه الفارغ لا يحمل معلومة تساعد النموذج، ويدخل ضجيجاً على التدريب.

**الطريقة:** عدّ القيم الفارغة (NaN) والنصوص الفارغة ونصوص المسافات فقط في كل جدول، ثم توزيع عدد الكلمات للأسئلة القصيرة وقراءة أمثلة منها.

**السبب:** فحصت القيم الفارغة قبل أي معالجة أخرى، لأن وجودها يؤثر على كل الخطوات التالية.

**النتائج:** لا توجد قيم NaN ولا نصوص فارغة ولا نصوص مسافات فقط في train ولا في validation (عمود الفئة غير موجود في test أصلاً). أما النصوص القصيرة فيوجد في train ثلاثة أسئلة من كلمة واحدة و26 سؤالاً من كلمتين فقط.

In [ ]:
missing = pd.DataFrame({name: df.isna().sum() for name, df in splits.items()})
missing

In [ ]:
for name, df in splits.items():
    text = df[Columns.TEXT].fillna("").astype(str)
    nan_count = df[Columns.TEXT].isna().sum()
    empty_count = (text == "").sum() - nan_count
    spaces_only = ((text != "") & (text.str.strip() == "")).sum()
    print(f"{name:<6} NaN={nan_count:>4}  فاضي={empty_count:>4}  مسافات فقط={spaces_only:>4}")

In [ ]:
words = train[Columns.TEXT].fillna("").astype(str).str.split().str.len()
words[words <= 10].value_counts().sort_index().rename("عدد الأسئلة")

In [ ]:
for n in range(1, 6):
    subset = train[words == n]
    print(f"--- أسئلة من {n} كلمات: {len(subset)} سؤال")
    display(subset[[Columns.ID, Columns.TEXT, Columns.LABEL]].head(8))

### القرار: عتبة النصوص القصيرة

الجدول التالي يبيّن كم سؤالاً يُحذف عند كل عتبة. القرار الحالي مكتوب في `DECISIONS["min_words"]` في ملف `quality.py`.

**السبب:** قرأت أمثلة الأسئلة القصيرة قبل اختيار العتبة. معظمها يحمل معلومة طبية واضحة مثل «علاج التسوس» و«ضغط العين» و«علاج الثعلبة» و«ما هو الانيميا». والسؤال الوحيد الذي لا يحمل معلومة هو «استفسار». كما أن عدّ الكلمات بالمسافات غير دقيق، فمثلاً «هل.حبوب.الدخن.مسموح.سكري.نوع.ثاني» يُحسب كلمة واحدة مع أنه سؤال كامل. ورفع العتبة يحذف أسئلة مفيدة كثيرة: عند العتبة 4 تُحذف 167 سؤالاً من train و29 من validation، وعند 5 تُحذف 604 و82.

**ملاحظة:** بعض الأسئلة القصيرة يبدو تصنيفها مشكوكاً فيه (مثل «الكرياتنين الطبيعي» بفئة صحة الأسنان، و«امراض نفسيه» بفئة ارتفاع ضغط الدم). لا يمكن تصحيح التصنيفات تلقائياً، لذلك لم أغيّرها.

**القرار:** `min_words = 1`، أي لا يُحذف شيء بهذه الخطوة حالياً لأنه لا توجد أسئلة فارغة. وتبقى الخطوة فعّالة في الكود إذا ظهرت أسئلة فارغة في بيانات جديدة.

In [ ]:
rows = []
for threshold in range(1, 6):
    row = {"min_words": threshold}
    for name in ("train", "val"):
        row[name] = int(fixer.find_short(splits[name], threshold).sum())
    rows.append(row)
pd.DataFrame(rows).set_index("min_words")

In [ ]:
print("القرار الحالي: min_words =", DECISIONS["min_words"])

## 2. أسماء الفئات

**المشكلة:** الفئة نفسها قد تكون مكتوبة بأكثر من شكل (بهمزة أو بدونها)، فيعدّها البرنامج فئات مختلفة.

**الطريقة:** مقارنة أسماء الفئات في البيانات مع ملف الترميز، ثم تجميع الأسماء التي تتطابق بعد توحيد الهمزات والمسافات، وقياس طول الأسئلة في كل شكل.

**السبب:** ملف الترميز فيه 10 فئات رسمية، وملف التسليم لا يقبل غير هذه الأسماء (حسب `submission.py`)، فيجب أن تطابقها أسماء الفئات في البيانات.

**النتائج:** في train وvalidation 14 اسماً مختلفاً بدل 10. خمسة أسماء في البيانات ليست في ملف الترميز، وكلها مكتوبة بدون همزة (امراض الاطفال، امراض الجهاز التنفسي، امراض الدم، امراض العيون، صحة الاسنان). والفئة الرسمية «أمراض العيون» لا تظهر في البيانات أصلاً، والموجود فيها «امراض العيون» فقط. أربع فئات تظهر بالشكلين (بدون همزة وبهمزة): الأطفال (5136 و2850)، الجهاز التنفسي (954 و663)، الدم (1334 و599)، الأسنان (2355 و922). بعد توحيد الهمزات والمسافات يتطابق كل اسم في البيانات مع اسم واحد من الملف.

**ملاحظة مهمة:** الشكلان ليسا متماثلين بالأسلوب. متوسط طول السؤال في الشكل بدون همزة حوالي 16–20 كلمة، وفي الشكل بهمزة حوالي 39–44 كلمة، وهذا يتكرر في الفئات الأربع. وأمثلة الشكل بهمزة أطول وأكثر رسمية. أرجّح أن الشكلين من مصدرين مختلفين لجمع البيانات، لكنه استنتاج من الأرقام وليس مؤكداً. ومعناه أن شكل كتابة الاسم يحمل معلومة عن المصدر، وهذه المعلومة تضيع بعد التوحيد.

**القرار:** توحيد كل الأسماء إلى الأسماء الرسمية من ملف الترميز، لأن ملف التسليم يتطلبها. النتيجة: 14 فئة صارت 10.

In [ ]:
label_map

In [ ]:
official = set(label_map[Columns.LABEL])

for name in ("train", "val"):
    used = set(splits[name][Columns.LABEL].dropna().unique())
    print(f"{name:<6} عدد الفئات بالبيانات: {len(used)}  |  بملف الترميز: {len(official)}")
    print("   بالبيانات وليست بملف الترميز:", sorted(used - official))
    print("   بملف الترميز وليست بالبيانات:", sorted(official - used))

In [ ]:
groups = fixer.label_groups(train)
groups

In [ ]:
words_per_label = train.assign(words=train[Columns.TEXT].fillna("").astype(str).str.split().str.len())
stats = words_per_label.groupby(Columns.LABEL)["words"].agg(["count", "mean", "median"]).round(1)
stats.loc[groups["label"]]

In [ ]:
for key, group in groups.groupby("key"):
    print("=" * 70)
    for label in group["label"]:
        subset = train[train[Columns.LABEL] == label]
        print(f"--- {label}  ({len(subset)} سؤال)")
        for question in subset[Columns.TEXT].sample(3, random_state=RANDOM_STATE):
            print("   *", str(question)[:120])

## 3. التكرار

**المشكلة:** السؤال المكرر يعطي النموذج وزناً زائداً، وإذا وقعت نسخة في train ونسخة في validation أصبح التقييم غير نزيه.

**الطريقة:** عدّ المكرر حرفياً، ثم المكرر بعد التطبيع (توحيد الهمزات والياء والتاء المربوطة وحذف الترقيم) لأنه يكشف تكراراً مخفياً. أما التشابه الجزئي (TF-IDF) فهو عند الشخص 1.

**النتائج:** في train 25 سؤالاً مكرراً حرفياً، ويرتفع العدد إلى 37 بعد التطبيع، أي 12 تكراراً إضافياً لا يظهر بالمطابقة الحرفية (مثل «قراءة» و«قراءه»، أو اختلاف بنقطة أو سطر جديد). في validation لا يوجد تكرار. من الـ 37: 33 نسخة بنفس الفئة تُحذف، و4 هي أسئلة بفئتين مختلفتين وتُعالج في قسم التضارب. وكثير من الأسئلة المكررة أرقام id لها متجاورة (مثل 393 و394، و50790 و50791)، فيبدو أن السؤال أُرسل مرتين.

**القرار:** حذف النسخ المكررة بعد التطبيع مع الإبقاء على أول نسخة (33 صفاً).

In [ ]:
for name in ("train", "val"):
    df = splits[name]
    exact = df.duplicated(Columns.TEXT, keep="first").sum()
    normalized = fixer.text_key(df[Columns.TEXT]).duplicated(keep="first").sum()
    print(f"{name:<6} مكرر حرفياً={exact:>5}  مكرر بعد التطبيع={normalized:>5}  (حجم الجدول {len(df)})")

In [ ]:
duplicates = fixer.find_duplicates(train)
print("عدد الصفوف المكررة بالـ train (مع النسخة الأولى):", len(duplicates))
duplicates[[Columns.ID, Columns.TEXT, Columns.LABEL]].head(20)

In [ ]:
raw_dup = train.duplicated(Columns.TEXT, keep=False)
key_dup = fixer.find_duplicates(train).index
hidden = train.loc[key_dup.difference(train.index[raw_dup])]
print("صفوف مكررة بعد التطبيع فقط (ما انكشفت بالمطابقة الحرفية):", len(hidden))
hidden[[Columns.ID, Columns.TEXT, Columns.LABEL]].head(10)

## 4. تضارب الملصقات

**المشكلة:** إذا ظهر السؤال نفسه بفئتين مختلفتين فأحد التصنيفين خاطئ على الأقل، والنموذج لا يستطيع تعلّم الاثنين معاً.

**الطريقة:** تجميع الأسئلة حسب نصها بعد التطبيع وبعد توحيد أسماء الفئات (حتى لا يُحسب اختلاف الهمزة في اسم الفئة تضارباً)، ثم عرض المجموعات التي فيها أكثر من فئة.

**النتائج:** 8 صفوف فقط (4 أسئلة) من أصل 58135. والأزواج: (الغدد الصماء، الطب العام) و(الطب العام، الدم) و(الغدد الصماء، ضغط الدم) و(الغدد الصماء، الجلدية). وهي حالات حدودية بين تخصصين، مثل سؤال تحليل الأنيميا (mchc) بين الطب العام والدم، وسؤال فحص السكر بين الطب العام والغدد الصماء.

**القرار:** حذف كل نسخ هذه الأسئلة (8 صفوف) لأنه لا توجد طريقة موثوقة لمعرفة الفئة الصحيحة، والعدد صغير جداً (حوالي 0.014%) فلا يؤثر على حجم البيانات.

In [ ]:
unified_train = fixer.unify_label_names(train)
conflicts = fixer.find_conflicts(unified_train)
print("عدد الصفوف المتضاربة:", len(conflicts), " | عدد الأسئلة المختلفة:", conflicts["_key"].nunique())
conflicts[[Columns.ID, Columns.TEXT, Columns.LABEL]].head(20)

In [ ]:
conflict_pairs = (
    conflicts.groupby("_key")[Columns.LABEL]
    .agg(lambda labels: " | ".join(sorted(set(labels))))
    .value_counts()
    .head(10)
    .rename("عدد الأسئلة")
)
conflict_pairs

## 5. الضجيج

**المشكلة:** النصوص غير العربية أو الرموز الغريبة قد تكون ضجيجاً، لكنها قد تحمل أيضاً معلومات طبية، فلا يصح حذفها قبل النظر في محتواها.

**الطريقة:** حساب نسبة الحروف العربية في كل سؤال، وعدّ الأنماط الغريبة، وقراءة أمثلة وعينة عشوائية من 30 سؤالاً.

**النتائج:** متوسط نسبة الحروف العربية 0.987 والوسيط 1.0. عدد الأسئلة التي نسبتها أقل من 0.2 هو 43، وأقل من 0.5 هو 191، وأقل من 0.8 هو 932. بقراءة الأمثلة وجدت أن معظمها نتائج تحاليل لُصقت بالإنجليزية (TSH وHGB وWBC وغيرها) أو تقارير أشعة، وبعضها أسئلة كاملة بالإنجليزية، وسؤال واحد بالعربية المكتوبة بأحرف لاتينية (bimada tansahoni...). الأحرف اللاتينية موجودة في 6330 سؤالاً (حوالي 10.9%)، والروابط في 7 فقط، والرموز الغريبة في 1337 سؤالاً، وتكرار الحرف 4 مرات فأكثر في 88 سؤالاً، ولا توجد أسئلة أرقام فقط. وفي الأمثلة التي قرأتها من العينة العشوائية لم أجد أسئلة غير طبية.

**القرار:** عدم حذف شيء (`min_arabic_ratio = 0`)، لأن أسماء التحاليل والوحدات بالإنجليزية معلومة مفيدة لتحديد الفئة (غدد، دم...)، والحالات الشاذة قليلة جداً. وتُحوَّل الأحرف اللاتينية إلى أحرف صغيرة في التنظيف حتى لا تُحسب «TSH» و«tsh» كلمتين.

In [ ]:
ratio = train[Columns.TEXT].map(arabic_ratio)
print(ratio.describe().round(3))
for threshold in (0.2, 0.5, 0.8):
    print(f"أسئلة نسبة العربي فيها أقل من {threshold}: {(ratio < threshold).sum()}")

In [ ]:
low_arabic = train[ratio < 0.5]
low_arabic[[Columns.ID, Columns.TEXT, Columns.LABEL]].head(15)

In [ ]:
text = train[Columns.TEXT].fillna("").astype(str)
patterns = {
    "روابط": r"https?://|www\.",
    "حروف لاتينية": r"[A-Za-z]",
    "رموز غريبة": r"[^\w\s\u0600-\u06FF.,؟?!،؛:()\-/%]",
    "حرف مكرر 4 مرات أو أكثر": r"([^\W\d_])\1{3,}",
    "أرقام فقط": r"^\s*[\d\s.,]+$",
}
pd.Series({name: text.map(lambda t: re.search(regex, t) is not None).sum() for name, regex in patterns.items()}, name="عدد الأسئلة")

In [ ]:
train[[Columns.ID, Columns.TEXT, Columns.LABEL]].sample(30, random_state=RANDOM_STATE)

## 6. التسريب بين الأقسام (قرار المعالجة)

**المشكلة:** وجود السؤال نفسه في train وفي validation يجعل نتيجة التقييم أعلى من الواقع لأن النموذج رأى السؤال أثناء التدريب. الحذف مسموح من train وvalidation فقط، وممنوع من test.

**الطريقة:** أرقام وأمثلة التسريب التفصيلية عند الشخص 1. هنا أحسب الأرقام نفسها بمفتاح التطبيع الخاص بي (الأشد من المطابقة الحرفية) لاتخاذ قرار المعالجة، ولذلك قد يختلف العدد قليلاً عن أرقامه.

**النتائج:** 9 أسئلة من validation (0.12%) موجودة في train، و7 أسئلة من test (0.10%) موجودة في train، ولا يوجد تسريب بين test وvalidation.

**القرار:** حذف الأسئلة التسعة من validation (`drop_leaked = True`) لأن الحذف مسموح منه وبذلك يصبح التقييم أنزه. أما أسئلة test السبعة فلا تُحذف لأن حذف أي صف من test ممنوع (يجب أن يبقى عدد صفوفه كما هو).

In [ ]:
leaked_val = fixer.find_leaked(val, train)
leaked_test_train = fixer.find_leaked(test, train)
leaked_test_val = fixer.find_leaked(test, val)
print(f"validation الموجود بـ train: {leaked_val.sum()} من {len(val)} ({leaked_val.mean():.2%})")
print(f"test الموجود بـ train:       {leaked_test_train.sum()} من {len(test)} ({leaked_test_train.mean():.2%})")
print(f"test الموجود بـ validation:  {leaked_test_val.sum()} من {len(test)} ({leaked_test_val.mean():.2%})")

In [ ]:
val[leaked_val][[Columns.ID, Columns.TEXT, Columns.LABEL]].head(10)

## 7. تطبيق القرارات

**الطريقة:** القرارات كلها مكتوبة في مكان واحد هو `DECISIONS` في `quality.py`، وتُنفَّذ بنفس القيم في الدفتر وفي دالة `apply_quality_fixes` التي يستدعيها الفريق، حتى لا يختلف ما في الدفتر عمّا يستخدمه الباقون. أسماء الفئات الرسمية تأتي من ملف الترميز.

**النتائج:** train: 58135 ← 58094 (حُذف 41 صفاً: 8 تضارب و33 تكرار). validation: 7267 ← 7258 (حُذف 9 بسبب التسريب). test: 7267 بدون حذف، وتحققت بـ `assert` أن عدده وترتيب id فيه كما هما، وأن كل الفئات بعد التنظيف ضمن ملف الترميز. عدد الفئات 14 ← 10.

**توزيع الفئات بعد التوحيد:** ست فئات بين 12.8% و13.7% (الأطفال والعيون والجلدية والطب العام والنسائية والغدد الصماء)، وأربع فئات أصغر: ضغط الدم 6.88% والأسنان 5.64% والدم 3.33% والتنفسي 2.78%. أي أن البيانات غير متوازنة، ولذلك مقياس macro-F1 أنسب من الدقة وحدها.

In [ ]:
fixer = QualityFixer(label_names=label_map[Columns.LABEL], **DECISIONS)
print(DECISIONS)

train_fixed = fixer.apply(train, is_train=True)
train_steps = fixer.summary()
val_fixed = fixer.apply(val, is_train=True, reference=train_fixed)
val_steps = fixer.summary()
test_fixed = fixer.apply(test, is_train=False)

assert len(test_fixed) == len(test), "ممنوع حذف أي صف من test"
assert list(test_fixed[Columns.ID]) == list(test[Columns.ID])
assert set(train_fixed[Columns.LABEL]) <= set(label_map[Columns.LABEL]), "في فئات خارج ملف الترميز"
assert set(val_fixed[Columns.LABEL]) <= set(label_map[Columns.LABEL]), "في فئات خارج ملف الترميز"
print("train:"); display(train_steps)
print("val:"); display(val_steps)

In [ ]:
def class_table(df):
    counts = df[Columns.LABEL].value_counts()
    return pd.DataFrame({"count": counts, "percent": (counts / len(df) * 100).round(2)})


before = class_table(train)
after = class_table(train_fixed)
before.join(after, lsuffix="_before", rsuffix="_after", how="outer").fillna(0)

In [ ]:
print("عدد الفئات قبل:", train[Columns.LABEL].nunique(), " بعد:", train_fixed[Columns.LABEL].nunique())
print("train:", len(train), "->", len(train_fixed), f"(حُذف {len(train) - len(train_fixed)})")
print("val:  ", len(val), "->", len(val_fixed), f"(حُذف {len(val) - len(val_fixed)})")
print("test: ", len(test), "->", len(test_fixed), "(بدون حذف)")

## 8. ملخص القرارات

| المشكلة | العدد | القرار | السبب |
|---|---|---|---|
| فارغ / NaN | 0 في train وvalidation | لا شيء للحذف (الخطوة جاهزة في الكود) | لا توجد قيم فارغة في البيانات الحالية |
| نصوص قصيرة | 3 بكلمة واحدة، 26 بكلمتين (train) | ترك (`min_words = 1`) | معظمها يحمل معلومة طبية، وعدّ الكلمات بالمسافات غير دقيق |
| أسماء الفئات | 14 اسماً ← 10 | توحيد إلى أسماء ملف الترميز | ملف التسليم يقبل الأسماء الرسمية فقط، والفرق في الهمزات والمسافات |
| تكرار حرفي | 25 في train، 0 في validation | حذف مع إبقاء أول نسخة | النسخة المكررة تعطي النموذج وزناً زائداً |
| تكرار بعد التطبيع | 37 في train (33 حُذفت و4 معالجة كتضارب) | حذف مع إبقاء أول نسخة | يكشف 12 تكراراً لا يظهر بالمطابقة الحرفية |
| تضارب الملصقات | 8 صفوف (4 أسئلة) | حذف كل النسخ | لا يمكن معرفة الفئة الصحيحة، والعدد صغير جداً |
| ضجيج | 191 سؤالاً نسبة العربية فيها أقل من 0.5 | ترك | معظمها نتائج تحاليل بالإنجليزية تحمل معلومات مفيدة |
| تسريب | 9 في validation، 7 في test | حذف التسعة من validation وترك test | تقييم أنزه، وحذف أي صف من test ممنوع |

# الجزء الثاني: تنظيف النص (`preprocess.py`)

هذا الجزء يستخدم `train_fixed` و`val_fixed` من قسم 7، لذلك يجب تشغيل الأقسام السابقة أولاً (Run All). لكل خطوة تنظيف: عدد الأسئلة التي تغيّرت، وحجم المفردات قبل وبعد، و10 أمثلة حقيقية من البيانات.

## 9. جدول قبل/بعد لكل خطوة

**المشكلة:** النص الخام فيه تشكيل وتطويل وهمزات مختلفة وترقيم وأرقام هندية وحروف مكررة، والنموذج يعدّ كل شكل كلمة مختلفة.

**الطريقة:** تُطبَّق كل خطوة وحدها (خيار واحد من `CleanOptions`) على `train_fixed`، ثم نقيس كم سؤالاً تغيّر وكم نقص حجم المفردات (عدد الكلمات المختلفة بعد التقسيم على المسافات)، ونعرض 10 أمثلة عشوائية من الأسئلة المتغيّرة. المقتطف المعروض يبدأ قرب أول اختلاف في السؤال.

**السبب:** أريد معرفة الأثر الحقيقي لكل خطوة على حدة، حتى أحدد ما يدخل في `light` وما يدخل في `heavy`.

**النتائج:** حجم المفردات الأصلي 126,359 كلمة.
- الخطوات الخفيفة لا تغيّر المعنى: التشكيل في 7.67% من الأسئلة ويقلل المفردات 1,345 كلمة، والتطويل في 0.33% (119 كلمة أقل تقريباً)، والرموز المخفية في 0.18% (منها علامات اتجاه الكتابة وحرف غير مرئي يقطع الكلمة من داخلها مثل «ه‍ل»). وتوحيد المسافات والأسطر يمسّ 28.66% من الأسئلة (بسبب `\r\n`) ولا يغيّر حجم المفردات لأنه لا يغيّر الكلمات نفسها.
- خطوات التوحيد: الألف في 42.72% من الأسئلة (4,880 كلمة أقل)، والياء ى←ي في 38.33% (1,593)، والتاء المربوطة ة←ه في 66.99% (4,307).
- أكبر أثر على المفردات لعلامات الترقيم: 27,305 كلمة أقل (حوالي 21.6%)، لأن الترقيم الملتصق بالكلمة («الحل؟») يجعلها كلمة مختلفة عن «الحل».
- الأرقام الهندية ← لاتينية في 15.32% من الأسئلة، وتكرار الحروف (مثل «خفييييف») في 0.56%، والأحرف الكبيرة ← صغيرة في 5.28% (مثل TSH وtsh).
- إخفاء الأرقام يطال أكثر من نصف الأسئلة. القاعدة تخفي الرقم المستقل والأرقام الملتصقة بحروف عربية («عمري34» و«٨شهور»)، وتترك الرموز اللاتينية التي فيها أرقام (HbA1c وB12 و10mg) لأنها أسماء تحاليل وأدوية وليست كميات.
- حذف الكلمات الشائعة يغيّر حوالي 96% من الأسئلة (لأن كل سؤال تقريباً فيه «في» أو «من» أو «هل»)، لكن أثره على المفردات بضع مئات من الكلمات فقط لأنها كلمات قليلة العدد كثيرة التكرار.

**القرار:** الخطوات الخفيفة تُطبَّق دائماً لأنها آمنة، وبقية الخطوات تدخل في `heavy` ويمكن تجربتها بشكل منفصل (قسم 12).

In [ ]:
from dataclasses import replace

from preprocess import CleanOptions, TextCleaner, apply_clean

SOURCE = train_fixed

NORMALIZED = CleanOptions(
    normalize_spaces=True, lowercase=True, normalize_alef=True, normalize_yeh=True,
    normalize_teh_marbuta=True, remove_punctuation=True,
)

STEPS = {
    "حذف الرموز المخفية": (CleanOptions(strip_invisible=True), None),
    "إزالة التشكيل": (CleanOptions(remove_diacritics=True), None),
    "إزالة التطويل": (CleanOptions(remove_tatweel=True), None),
    "المسافات والأسطر": (CleanOptions(normalize_spaces=True), None),
    "توحيد الألف (أ إ آ ← ا)": (CleanOptions(normalize_alef=True), None),
    "ى ← ي": (CleanOptions(normalize_yeh=True), None),
    "ة ← ه": (CleanOptions(normalize_teh_marbuta=True), None),
    "علامات الترقيم": (CleanOptions(remove_punctuation=True, normalize_spaces=True), None),
    "الأرقام الهندية ← لاتينية": (CleanOptions(normalize_digits=True), None),
    "إخفاء الأرقام (NUM)": (CleanOptions(normalize_digits=True, mask_numbers=True, normalize_spaces=True), None),
    "تكرار الحروف (3 مرات أو أكثر)": (CleanOptions(collapse_repeats=True), None),
    "الحروف اللاتينية الكبيرة ← صغيرة": (CleanOptions(lowercase=True), None),
    "حذف الكلمات الشائعة": (replace(NORMALIZED, remove_stopwords=True), NORMALIZED),
}


def vocab_size(texts):
    return len(set(" ".join(texts).split()))


def snippet(before, after, width=45):
    first_diff = next((i for i, (a, b) in enumerate(zip(before, after)) if a != b), min(len(before), len(after)))
    start = max(0, first_diff - width)
    if start:
        start = before.rfind(" ", 0, start + 1) + 1
    return before[start:first_diff + width], after[start:first_diff + width]


def step_report(options, base=None, n=10):
    before = SOURCE[Columns.TEXT].fillna("").astype(str)
    if base is not None:
        before = before.map(TextCleaner(options=base).clean)
    after = SOURCE[Columns.TEXT].map(TextCleaner(options=options).clean)
    changed = before != after
    picked = before[changed].sample(min(n, int(changed.sum())), random_state=RANDOM_STATE).index
    pairs = [snippet(before[i], after[i]) for i in picked]
    examples = pd.DataFrame(pairs, columns=["قبل", "بعد"], index=picked)
    summary = {
        "rows_changed": int(changed.sum()),
        "percent": round(changed.mean() * 100, 2),
        "vocab_before": vocab_size(before),
        "vocab_after": vocab_size(after),
    }
    return summary, examples


summaries, examples = {}, {}
for name, (options, base) in STEPS.items():
    summaries[name], examples[name] = step_report(options, base)

step_table = pd.DataFrame(summaries).T
step_table["vocab_change"] = step_table["vocab_after"] - step_table["vocab_before"]
step_table

In [ ]:
for name, table in examples.items():
    print("=" * 100)
    print(f"{name}  |  تغيّر {summaries[name]['rows_changed']} سؤال ({summaries[name]['percent']}%)")
    display(table)

## 10. light مقابل heavy

**الطريقة:** تشغيل `none` و`light` و`heavy` على `train_fixed` ومقارنة حجم المفردات ومتوسط عدد الكلمات وعدد الأسئلة التي صارت فارغة، ثم عرض 5 أسئلة كاملة بالنسخ الثلاث.

**السبب:** أردت أن أعرف كم يتغيّر النص فعلاً بين الأوضاع الثلاثة قبل المقارنة بالأداء.

**النتائج:** حجم المفردات: `none` = 126,359 و`light` = 124,781 (نقص 1.25% فقط) و`heavy` حوالي 85 ألفاً (نقص حوالي 32%). متوسط عدد الكلمات في السؤال 26.8 في `none` و`light`، وحوالي 21 في `heavy`. لا يوجد أي سؤال صار فارغاً بعد التنظيف في أي وضع، وفي train وvalidation وtest. في الأمثلة الخمسة: `light` يزيل التشكيل والتطويل فقط ويبقي النص كما كتبه صاحبه، أما `heavy` فيحذف كلمات مثل «هل» و«في»، ويوحّد الهمزات، ويحوّل الأرقام إلى `NUM`.

In [ ]:
mode_rows = []
cleaned = {}
for mode in ("none", "light", "heavy"):
    out = apply_clean(train_fixed, mode)
    cleaned[mode] = out
    texts = out[Columns.CLEAN]
    mode_rows.append({
        "mode": mode,
        "vocab": vocab_size(texts),
        "mean_words": round(texts.str.split().str.len().mean(), 1),
        "empty_after_clean": int((texts.str.strip() == "").sum()),
    })
pd.DataFrame(mode_rows).set_index("mode")

In [ ]:
picked = train_fixed.sample(5, random_state=RANDOM_STATE).index
for i in picked:
    print("-" * 100)
    print("الأصلي:", train_fixed.loc[i, Columns.TEXT])
    for mode in ("light", "heavy"):
        print(f"{mode:<6}:", cleaned[mode].loc[i, Columns.CLEAN])

In [ ]:
for name, df in (("train_fixed", train_fixed), ("val_fixed", val_fixed), ("test_fixed", test_fixed)):
    for mode in ("light", "heavy"):
        out = apply_clean(df, mode)
        assert len(out) == len(df) and out[Columns.CLEAN].notna().all()
        print(f"{name:<12} {mode:<6} أسئلة فاضية بعد التنظيف: {int((out[Columns.CLEAN].str.strip() == '').sum())}")

## 11. مراجعة الكلمات الشائعة (Stop words) يدوياً

**المشكلة:** قائمة NLTK العربية عامة وليست طبية، وفيها كلمات قد تحمل معلومة طبية (نفي، زمن، أو كلمات ترتبط بفئة معينة).

**الطريقة:** (أ) أكثر الكلمات المحذوفة فعلاً تكراراً في البيانات. (ب) لكل كلمة في قائمة NLTK الأصلية: عدد ظهورها، والفئة التي تظهر فيها أكثر، ونسبة ظهورها في تلك الفئة مقارنة بحجم الفئة (`lift`)، والعمود `kept` يبيّن هل استثنيتها من الحذف. (ج) أمثلة حقيقية لبعض الكلمات المرشحة. (د) قائمة الكلمات المستثناة.

**السبب:** الكلمة الشائعة التي تتركز في فئة واحدة قد تكون معلومة وليست حشواً، فحذفها يضيّع إشارة مفيدة.

**النتائج:** أكثر الكلمات المحذوفة تكراراً هي كلمات الربط والاستفهام العادية: «في» (38,798) و«من» (36,481) و«هل» (22,466) و«أنا» و«مع» و«أو»، وهذه لا تحمل معلومة طبية. و«على» تظهر «علي» لأن التنظيف يوحّد ى←ي، وتُطبَّق على قائمة الكلمات الشائعة نفس التوحيد حتى تتطابق مع النص.

وفي جدول `lift` ظهرت كلمات من القائمة ترتبط بفئة محددة، فاستثنيتها من الحذف في `KEEP_WORDS`:
- **أسماء الأشهر** (أبريل، مايو، فبراير...): من 69% إلى 93% من ظهورها في الأسئلة النسائية (مقابل 14% لحجم الفئة)، ويبدو أنها تواريخ الدورة والحمل.
- **الأعداد والترتيب** (خامس، ثالث، تسع، ثمانية...): «خامس» 91% في النسائية، و«تسع» و«ثمانيه» في أسئلة الأطفال، ويبدو أنها عمر الطفل بالأشهر وشهر الحمل.
- **«د» و«دال»**: ترتبط بالطب العام (55% و64%)، ويبدو أنها «فيتامين د».
- **«اري» و«كلتا» و«امام»**: ترتبط بأسئلة العيون («أرى»، «كلتا عيني»، «أمام عيني»).
- **النفي** (لا، لم، بدون، ولا، وما...): أبقيته لأن النفي يغيّر معنى الشكوى («لا يوجد ألم» تختلف عن «يوجد ألم»).

التفسيرات أعلاه استنتاجات من الأرقام، وقد تحققت منها بقراءة أمثلة الكلمات في الخلية الثالثة.

**القرار:** حذف قائمة NLTK مع استثناء الكلمات السابقة، والقائمة النهائية قابلة للتعديل في `KEEP_WORDS` في `preprocess.py`.

In [ ]:
from nltk.corpus import stopwords as nltk_stopwords

from preprocess import KEEP_WORDS

heavy_options = CleanOptions.for_mode("heavy")
normalizer = TextCleaner(options=replace(heavy_options, remove_stopwords=False))
final_stopwords = TextCleaner("heavy").stopwords
nltk_words = {normalizer.clean(word) for word in nltk_stopwords.words("arabic")} - {""}
kept_words = {normalizer.clean(word) for word in KEEP_WORDS}
print("قائمة NLTK بعد التوحيد:", len(nltk_words), "| المستثناة:", len(nltk_words & kept_words), "| المحذوفة فعلاً:", len(final_stopwords))

tokens = train_fixed[Columns.TEXT].map(normalizer.clean).str.split()
exploded = pd.DataFrame({"word": tokens, Columns.LABEL: train_fixed[Columns.LABEL]}).explode("word")
exploded = exploded[exploded["word"].isin(nltk_words)]
word_by_class = exploded.groupby(["word", Columns.LABEL]).size().unstack(fill_value=0)
word_totals = word_by_class.sum(axis=1).rename("count").sort_values(ascending=False)
word_totals[word_totals.index.isin(final_stopwords)].head(40).to_frame()

In [ ]:
class_share = train_fixed[Columns.LABEL].value_counts(normalize=True)
top_class = word_by_class.idxmax(axis=1)
top_class_share = word_by_class.max(axis=1) / word_by_class.sum(axis=1)
skew = pd.DataFrame({
    "count": word_totals,
    "top_class": top_class,
    "share_in_top_class": top_class_share.round(2),
    "class_prior": top_class.map(class_share).round(2),
})
skew["lift"] = (skew["share_in_top_class"] / skew["class_prior"]).round(2)
skew["kept"] = skew.index.isin(kept_words)
skew[skew["count"] >= 30].sort_values("lift", ascending=False).head(40)

In [ ]:
candidates = ["مارس", "خامس", "تسع", "دال", "اري", "كلتا"]
for word in candidates:
    mask = tokens.map(lambda words: word in words)
    print("=" * 80)
    print(f"{word}: {int(mask.sum())} سؤالاً")
    for question in train_fixed.loc[mask, Columns.TEXT].head(3):
        print("   *", question[:110].replace("\n", " "))

In [ ]:
print("الكلمات المستثناة من الحذف:", sorted(kept_words))

## 12. تجربة سريعة: ى/ي وة/ه ومقارنة الأوضاع

**الطريقة:** TF-IDF (يتعلّم على `train_fixed` فقط) مع LinearSVC، والتقييم على `val_fixed` (بعد حذف التسريب) بنسخ تنظيف مختلفة. هذا قياس سريع لتوثيق نتيجة ى/ي وة/ه (اللتان تعتبرهما الخطة اختياريتين) ولمقارنة الأوضاع، وتجارب الشخص 3 هي الأساس.

**السبب:** أردت أن أقيس بالأرقام هل التنظيف الثقيل يفيد فعلاً، وأن أوثّق أثر ى/ي وة/ه قبل أن أقرر إدخالهما في `heavy`.

**النتائج:** (القيم تقريبية من تشغيلي الأول، وتتغير أرقام `heavy` بحوالي 0.002 بعد إضافة الاستثناءات إلى `KEEP_WORDS`، فيجب مطابقتها مع جدول الخلية قبل التسليم.)
- `none`: macro-F1 حوالي 0.832 والدقة 0.855، و`light`: 0.832 و0.856. الفرق أقل من 0.001، أي أن التنظيف الخفيف لا يضر ولا يفيد بشكل ملموس مع هذا النموذج.
- `heavy`: macro-F1 حوالي 0.827 والدقة 0.851، أي أقل بحوالي 0.005 من `none` و`light`، مع أنه يخفض عدد الخصائص من حوالي 98 ألفاً إلى حوالي 85 ألفاً.
- `heavy` بدون ى/ي وة/ه: حوالي 0.825، أي أن توحيدهما يحسّن بمقدار صغير جداً (حوالي 0.002).
- `heavy` بدون حذف الكلمات الشائعة: حوالي 0.826، أي أن حذفها يحسّن بمقدار صغير جداً (حوالي 0.001).

**تحفظات:** كل الفروق أقل من نقطة مئوية، وهي من تشغيل واحد وتقسيم validation واحد، فلا أستطيع القول إن أياً منها فرق حقيقي. والنموذج بسيط وقد تختلف النتيجة مع نماذج أخرى (تجارب الشخص 3). وقد يكون سبب انخفاض `heavy` أن إخفاء الأرقام وحذف الكلمات يزيل معلومات مفيدة، لكنها فرضية لم أختبرها.

**القرار:** `light` هو الوضع الافتراضي المقترح للمرحلة الأولى لأنه آمن ونتيجته مساوية لـ `none`، و`heavy` يبقى متاحاً بخياراته المنفصلة لتجارب الفريق.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.svm import LinearSVC

heavy_options = CleanOptions.for_mode("heavy")
CONFIGS = {
    "none": CleanOptions.for_mode("none"),
    "light": CleanOptions.for_mode("light"),
    "heavy بدون ى/ي وة/ه": replace(heavy_options, normalize_yeh=False, normalize_teh_marbuta=False),
    "heavy بدون الكلمات الشائعة": replace(heavy_options, remove_stopwords=False),
    "heavy": heavy_options,
}

results = []
for name, options in CONFIGS.items():
    cleaner = TextCleaner(options=options)
    x_train = train_fixed[Columns.TEXT].map(cleaner.clean)
    x_val = val_fixed[Columns.TEXT].map(cleaner.clean)
    vectorizer = TfidfVectorizer()
    train_matrix = vectorizer.fit_transform(x_train)
    val_matrix = vectorizer.transform(x_val)
    model = LinearSVC(random_state=RANDOM_STATE).fit(train_matrix, train_fixed[Columns.LABEL])
    predictions = model.predict(val_matrix)
    results.append({
        "config": name,
        "features": train_matrix.shape[1],
        "accuracy": round(accuracy_score(val_fixed[Columns.LABEL], predictions), 4),
        "macro_f1": round(f1_score(val_fixed[Columns.LABEL], predictions, average="macro"), 4),
    })
pd.DataFrame(results).set_index("config")

## 13. ملخص قرارات التنظيف

| الخطوة | نسبة الأسئلة المتأثرة | light | heavy | السبب |
|---|---|---|---|---|
| التشكيل والتطويل والرموز المخفية | 7.67% و0.33% و0.18% | نعم | نعم | لا تغيّر المعنى وتقلل المفردات |
| المسافات والأسطر | 28.66% | نعم | نعم | توحيد `\r\n` والمسافات، ولا تغيّر المفردات |
| توحيد الألف | 42.72% | لا | نعم | يقلل 4,880 كلمة، لكنه قد يدمج كلمتين مختلفتين (أمر/امر) |
| ى←ي وة←ه | 38.33% و66.99% | لا | نعم | يقللان المفردات (1,593 و4,307)، وفرق الأداء صغير جداً |
| علامات الترقيم | 66.34% | لا | نعم | أكبر تقليل للمفردات (حوالي 21.6%) |
| الأرقام | الهندية 15.32%، والإخفاء أكثر من نصف الأسئلة | لا | نعم | توحيد الأرقام وتقليل الكلمات، مع إبقاء رموز مثل HbA1c وB12 |
| تكرار الحروف | 0.56% | لا | نعم | يوحّد «خفييييف» مع «خفيف» |
| الكلمات الشائعة | حوالي 96% | لا | نعم | بعد استثناء النفي والأشهر والأعداد وغيرها (`KEEP_WORDS`)، وأثرها على الأداء صغير |